<a href="https://colab.research.google.com/github/ABELCANTRIZZ/Kompuntasi-Numerik-UTS/blob/main/Metode__SPL_3_Variabel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SPL 3 Variabel - Eliminasi Gauss dengan Pivoting Parsial
Menyelesaikan sistem:

a11 x + a12 y + a13 z = b1
a21 x + a22 y + a23 z = b2
a31 x + a32 y + a33 z = b3

**Ide:** ubah matriks augmented [A|b] menjadi bentuk segitiga atas (eliminasi maju), lalu hitung z, y, x dari bawah ke atas (substitusi mundur).

## Cell 1 - Import library
`numpy` dipakai untuk array/matriks; ini satu-satunya library yang dibutuhkan.

In [ ]:
import numpy as np

## Cell 2 - Input sistem
`A` adalah matriks koefisien 3x3, `b` adalah vektor ruas kanan. Ganti angkanya untuk soal lain.
`dtype=float` memastikan pembagian tidak dibulatkan menjadi bilangan bulat.

In [ ]:
A = np.array([[2, 1, -1],
              [-3, -1, 2],
              [-2, 1, 2]], dtype=float)
b = np.array([8, -11, -3], dtype=float)

## Cell 3 - Fungsi eliminasi Gauss
Langkah di dalam fungsi:
1. Gabungkan A dan b menjadi matriks augmented `M` (3x4).
2. Untuk tiap kolom pivot `k`: pilih baris dengan nilai mutlak terbesar (pivoting parsial) agar pembagian lebih stabil, lalu tukar baris.
3. Jika pivot hampir nol, sistem tidak punya solusi tunggal.
4. Nolkan elemen di bawah pivot dengan operasi baris.
5. Substitusi mundur dari baris terakhir ke atas.

In [ ]:
def gauss_3var(A, b, tol=1e-12):
    n = len(b)
    M = np.hstack([A, b.reshape(-1, 1)])        # matriks augmented [A|b]

    for k in range(n - 1):
        p = k + np.argmax(np.abs(M[k:, k]))      # indeks baris dengan pivot terbesar
        if abs(M[p, k]) < tol:
            raise ValueError("Pivot nol: tidak ada solusi tunggal")
        if p != k:
            M[[k, p]] = M[[p, k]]                # tukar baris k dan p
        for i in range(k + 1, n):
            faktor = M[i, k] / M[k, k]           # pengali eliminasi
            M[i, k:] -= faktor * M[k, k:]        # baris i dikurangi faktor * baris k

    if abs(M[n - 1, n - 1]) < tol:
        raise ValueError("Pivot nol: tidak ada solusi tunggal")

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):               # substitusi mundur
        x[i] = (M[i, n] - M[i, i + 1:n] @ x[i + 1:]) / M[i, i]
    return x, M

## Cell 4 - Jalankan dan tampilkan hasil
Matriks segitiga atas ditampilkan agar proses eliminasi terlihat.

In [ ]:
x, segitiga = gauss_3var(A, b)
print("Matriks setelah eliminasi:\n", np.round(segitiga, 4))
print("\nx =", round(x[0], 6))
print("y =", round(x[1], 6))
print("z =", round(x[2], 6))

## Cell 5 - Verifikasi
Substitusi hasil ke A·x dan bandingkan dengan b, serta dengan solusi bawaan `np.linalg.solve`.

In [ ]:
print("A @ x      =", A @ x)
print("b           =", b)
print("Cocok dengan numpy:", np.allclose(x, np.linalg.solve(A, b)))